# Multimodal Hierarchical News Classification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vittoriacrugnolaa/University_projects/blob/main/Multimodal_News_Classification/notebooks/multimodal_news_classification.ipynb)

This project predicts hierarchical topic labels from three complementary inputs:

- raw article text, encoded by an embedding and a bidirectional LSTM;
- precomputed Bag-of-Words features, normalized and compressed with Truncated SVD;
- publication year and month, standardized and one-hot encoded.

The three learned representations are fused in a shared neural network. Evaluation includes a linear baseline, validation-only threshold tuning, and an optional hierarchy-consistent decoder that restricts predictions to label patterns observed in the training data.

## Reproducibility and leakage controls

All random generators use the same seed. Data is split with iterative multi-label stratification, and every learned preprocessing step is fitted on the training split only. The validation split is used for early stopping and threshold selection; the test split is evaluated once at the end.

The dataset archive is not stored in the repository because no redistribution license was supplied with the article text. The notebook verifies the exact archive checksum before loading its pickle payload.

In [ ]:
# Environment setup: works from a local clone or directly in Google Colab.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Vittoriacrugnolaa/University_projects.git"
PROJECT_DIRECTORY = "Multimodal_News_Classification"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    repository_root = Path("/content/University_projects")
    if not (repository_root / PROJECT_DIRECTORY).exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(repository_root)],
            check=True,
        )
    PROJECT_ROOT = repository_root / PROJECT_DIRECTORY
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "iterative-stratification>=0.1.9,<1",
        ],
        check=True,
    )
else:
    PROJECT_ROOT = next(
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / "pyproject.toml").exists()
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from news_classifier.data import load_dataset, prepare_features, split_dataset
from news_classifier.evaluation import (
    decode_to_valid_patterns,
    multilabel_metrics,
    per_label_metrics,
    tune_per_label_thresholds,
)
from news_classifier.model import (
    build_model,
    build_text_vectorizer,
    set_global_determinism,
    training_callbacks,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

SEED = 213
BOW_COMPONENTS = 256
MAX_TOKENS = 20_000
SEQUENCE_LENGTH = 80
BATCH_SIZE = 64
MAX_EPOCHS = 30

set_global_determinism(SEED)
print("TensorFlow:", tf.__version__)

## 1. Load and validate the data

For local execution, place `input_data.zip` in `data/`. In Colab, the upload dialog appears automatically when the archive is missing. The loader validates both the SHA-256 checksum and the expected data schema before deserializing the file.

In [ ]:
DATA_ARCHIVE = Path(
    os.environ.get("NEWS_DATA_ARCHIVE", PROJECT_ROOT / "data" / "input_data.zip")
)

if not DATA_ARCHIVE.exists() and IN_COLAB:
    from google.colab import files

    print("Upload the original input_data.zip archive.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No dataset archive was uploaded.")
    DATA_ARCHIVE = Path(next(iter(uploaded)))

if not DATA_ARCHIVE.exists():
    raise FileNotFoundError(
        f"Dataset archive not found at {DATA_ARCHIVE}. "
        "See data/README.md for setup instructions."
    )

load_start = time.perf_counter()
dataset = load_dataset(DATA_ARCHIVE)
print(f"Loaded {len(dataset.texts):,} samples in {time.perf_counter() - load_start:.1f}s")
print("BoW shape:", dataset.bow.shape, "| non-zero values:", f"{dataset.bow.nnz:,}")
print("Target shape:", dataset.labels.shape)

## 2. Exploratory analysis

In [ ]:
word_counts = np.fromiter(
    (len(text.split()) for text in dataset.texts),
    dtype=np.int32,
    count=len(dataset.texts),
)
label_support = dataset.labels.sum(axis=0).astype(int)
label_cardinality = dataset.labels.sum(axis=1)
valid_patterns = np.unique(dataset.labels.astype(np.int8), axis=0)

summary = pd.Series(
    {
        "samples": len(dataset.texts),
        "labels": dataset.labels.shape[1],
        "valid label patterns": len(valid_patterns),
        "mean labels per sample": label_cardinality.mean(),
        "median words per article": np.median(word_counts),
        "99th percentile words": np.percentile(word_counts, 99),
        "publication year range": f"{dataset.years.min()}–{dataset.years.max()}",
    },
    name="value",
)
display(summary.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(word_counts, bins=30, ax=axes[0], color="#4C78A8")
axes[0].axvline(SEQUENCE_LENGTH, color="#E45756", linestyle="--", label="sequence length")
axes[0].set(title="Article length", xlabel="Words", ylabel="Articles")
axes[0].legend()

sns.barplot(x=np.arange(len(label_support)), y=label_support, ax=axes[1], color="#72B7B2")
axes[1].set(title="Label support", xlabel="Label index", ylabel="Positive samples")
plt.tight_layout()
plt.show()

## 3. Iterative multi-label split

Iterative stratification preserves marginal label frequencies across the 70% training, 15% validation, and 15% test splits. Unlike stratifying on concatenated label strings, it also remains robust when a dataset contains rare label combinations.

In [ ]:
splits = split_dataset(dataset.labels, random_state=SEED)

split_table = pd.DataFrame(
    {
        "samples": [len(splits.train), len(splits.validation), len(splits.test)],
        "mean labels per sample": [
            dataset.labels[index].sum(axis=1).mean()
            for index in (splits.train, splits.validation, splits.test)
        ],
    },
    index=["train", "validation", "test"],
)
display(split_table)

label_prevalence = pd.DataFrame(
    {
        "train": dataset.labels[splits.train].mean(axis=0),
        "validation": dataset.labels[splits.validation].mean(axis=0),
        "test": dataset.labels[splits.test].mean(axis=0),
    }
)
print("Maximum label-prevalence difference:", f"{label_prevalence.max(axis=1).sub(label_prevalence.min(axis=1)).max():.4f}")

## 4. Numerical preprocessing

The 10,000-dimensional BoW matrix is only about 0.22% non-zero. It is therefore kept sparse, normalized row-wise, and reduced to a compact latent representation with Truncated SVD. This avoids the large dense input layer used by the original prototype while retaining broad lexical information.

Year is standardized, and month is one-hot encoded. Both preprocessors are fitted only on training rows.

In [ ]:
preprocess_start = time.perf_counter()
features = prepare_features(
    dataset,
    splits,
    bow_components=BOW_COMPONENTS,
    random_state=SEED,
)
print(f"Preprocessing completed in {time.perf_counter() - preprocess_start:.1f}s")
print("Reduced BoW:", features.bow_train.shape)
print("Metadata:", features.metadata_train.shape)
print(
    "Explained variance retained by SVD:",
    f"{features.bow_preprocessor.named_steps['reduce'].explained_variance_ratio_.sum():.1%}",
)

## 5. Linear baseline

A one-vs-rest logistic regression on the reduced BoW and metadata features provides a transparent benchmark. Thresholds are optimized independently for each label using validation data only.

In [ ]:
X_baseline_train = np.column_stack(
    (features.bow_train, features.metadata_train)
)
X_baseline_validation = np.column_stack(
    (features.bow_validation, features.metadata_validation)
)
X_baseline_test = np.column_stack(
    (features.bow_test, features.metadata_test)
)

baseline = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2_000,
        class_weight="balanced",
        solver="liblinear",
        random_state=SEED,
    ),
    n_jobs=1,
)
baseline.fit(X_baseline_train, dataset.labels[splits.train])

baseline_validation_probability = baseline.predict_proba(X_baseline_validation)
baseline_thresholds = tune_per_label_thresholds(
    dataset.labels[splits.validation],
    baseline_validation_probability,
)
baseline_test_probability = baseline.predict_proba(X_baseline_test)
baseline_test_prediction = (
    baseline_test_probability >= baseline_thresholds
).astype(np.int8)
baseline_pattern_prediction = decode_to_valid_patterns(
    baseline_test_probability,
    np.unique(dataset.labels[splits.train].astype(np.int8), axis=0),
)

pd.DataFrame(
    {
        "thresholded": multilabel_metrics(
            dataset.labels[splits.test], baseline_test_prediction
        ),
        "valid-pattern decoder": multilabel_metrics(
            dataset.labels[splits.test], baseline_pattern_prediction
        ),
    }
).T

## 6. Multimodal neural network

```text
raw text ── TextVectorization ── Embedding ── BiLSTM ───────┐
                                                            │
BoW counts ── L1 normalization ── Truncated SVD ── Dense ───┼─ Concatenate ─ Dense ─ Sigmoid(18)
                                                            │
year/month ── scaling + one-hot encoding ── Dense ──────────┘
```

The text vectorizer is adapted on training articles only and embedded inside the saved Keras model. Early stopping restores the weights with the lowest validation loss, while learning-rate reduction stabilizes late training.

In [ ]:
text_vectorizer = build_text_vectorizer(
    dataset.texts[splits.train],
    max_tokens=MAX_TOKENS,
    sequence_length=SEQUENCE_LENGTH,
)

model = build_model(
    text_vectorizer=text_vectorizer,
    bow_dimension=features.bow_train.shape[1],
    metadata_dimension=features.metadata_train.shape[1],
    number_of_labels=dataset.labels.shape[1],
)
model.summary()

In [ ]:
train_inputs = {
    "text": dataset.texts[splits.train].astype(object),
    "bow": features.bow_train,
    "metadata": features.metadata_train,
}
validation_inputs = {
    "text": dataset.texts[splits.validation].astype(object),
    "bow": features.bow_validation,
    "metadata": features.metadata_validation,
}

training_start = time.perf_counter()
history = model.fit(
    train_inputs,
    dataset.labels[splits.train],
    validation_data=(validation_inputs, dataset.labels[splits.validation]),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=training_callbacks(patience=4),
    verbose=2,
)
training_minutes = (time.perf_counter() - training_start) / 60
print(f"Training time: {training_minutes:.1f} minutes")

In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_frame[["loss", "val_loss"]].plot(ax=axes[0])
axes[0].set(title="Binary cross-entropy", xlabel="Epoch", ylabel="Loss")
history_frame[["macro_pr_auc", "val_macro_pr_auc"]].plot(ax=axes[1])
axes[1].set(title="Macro PR AUC", xlabel="Epoch", ylabel="Area under PR curve")
plt.tight_layout()
plt.show()

## 7. Validation-driven decision rules and final test evaluation

Two decision rules are compared:

1. **Per-label thresholds** maximize each label's validation F1 score.
2. **Valid-pattern decoding** selects the most likely hierarchical label combination among those observed in the training set.

The second rule is appropriate here because the target matrix represents a fixed hierarchy and contains only 11 valid paths. It prevents contradictory or structurally impossible outputs.

In [ ]:
validation_probability = model.predict(validation_inputs, verbose=0)
thresholds = tune_per_label_thresholds(
    dataset.labels[splits.validation],
    validation_probability,
)
training_patterns = np.unique(
    dataset.labels[splits.train].astype(np.int8),
    axis=0,
)

test_inputs = {
    "text": dataset.texts[splits.test].astype(object),
    "bow": features.bow_test,
    "metadata": features.metadata_test,
}
test_probability = model.predict(test_inputs, verbose=0)
threshold_prediction = (test_probability >= thresholds).astype(np.int8)
pattern_prediction = decode_to_valid_patterns(test_probability, training_patterns)

comparison = pd.DataFrame(
    {
        "linear baseline": multilabel_metrics(
            dataset.labels[splits.test], baseline_pattern_prediction
        ),
        "neural / tuned thresholds": multilabel_metrics(
            dataset.labels[splits.test], threshold_prediction
        ),
        "neural / valid-pattern decoder": multilabel_metrics(
            dataset.labels[splits.test], pattern_prediction
        ),
    }
).T
display(comparison.round(4))

In [ ]:
label_report = per_label_metrics(
    dataset.labels[splits.test],
    pattern_prediction,
    test_probability,
)
display(label_report.round(3))

fig, ax = plt.subplots(figsize=(12, 4))
plot_frame = label_report.melt(
    id_vars="label",
    value_vars=["f1", "average_precision"],
    var_name="metric",
    value_name="score",
)
sns.barplot(data=plot_frame, x="label", y="score", hue="metric", ax=ax)
ax.set(title="Per-label test performance", xlabel="", ylabel="Score", ylim=(0, 1.05))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8. Save the reproducible pipeline

The Keras model contains the text vectorizer and vocabulary. The companion preprocessing artifact contains the BoW reducer, metadata transformer, decision thresholds, and valid hierarchy paths. Generated artifacts are ignored by Git because they can be reproduced by running this notebook.

In [ ]:
ARTIFACTS_DIRECTORY = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIRECTORY.mkdir(exist_ok=True)

model.save(ARTIFACTS_DIRECTORY / "multimodal_news_classifier.keras")
joblib.dump(
    {
        "bow_preprocessor": features.bow_preprocessor,
        "metadata_preprocessor": features.metadata_preprocessor,
        "thresholds": thresholds,
        "valid_patterns": training_patterns,
    },
    ARTIFACTS_DIRECTORY / "preprocessing.joblib",
)

metrics_payload = {
    model_name: {metric: float(value) for metric, value in row.items()}
    for model_name, row in comparison.to_dict(orient="index").items()
}
with (ARTIFACTS_DIRECTORY / "metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, indent=2)

print("Saved model and preprocessing artifacts to", ARTIFACTS_DIRECTORY)

## 9. Inference example

The source dataset does not include the vocabulary used to construct its 10,000 BoW columns. Consequently, inference on a new article requires a BoW vector produced by the same external vocabulary. The raw-text branch remains fully self-contained in the saved model; this helper makes the remaining contract explicit rather than silently generating incompatible features.

In [ ]:
def predict_article(text, bow_vector, year, month):
    if sparse.issparse(bow_vector):
        bow_matrix = bow_vector.tocsr()
    else:
        bow_matrix = sparse.csr_matrix(np.asarray(bow_vector).reshape(1, -1))
    reduced_bow = features.bow_preprocessor.transform(bow_matrix).astype(np.float32)
    encoded_metadata = features.metadata_preprocessor.transform(
        np.array([[year, month]])
    ).astype(np.float32)
    probability = model.predict(
        {
            "text": np.asarray([text], dtype=object),
            "bow": reduced_bow,
            "metadata": encoded_metadata,
        },
        verbose=0,
    )
    prediction = decode_to_valid_patterns(probability, training_patterns)[0]
    return {
        "active_label_indices": np.flatnonzero(prediction).tolist(),
        "label_probabilities": probability[0].round(4).tolist(),
    }


example_index = splits.test[0]
example_prediction = predict_article(
    dataset.texts[example_index],
    dataset.bow[example_index],
    int(dataset.years[example_index]),
    int(dataset.months[example_index]),
)

print("Text:", dataset.texts[example_index][:180], "...")
print("True labels:", np.flatnonzero(dataset.labels[example_index]).tolist())
print("Predicted labels:", example_prediction["active_label_indices"])

## Conclusions

This implementation treats the dataset as a genuine multimodal and hierarchical learning problem. Sparse dimensionality reduction makes the BoW branch practical, the split and threshold selection avoid leakage, the baseline makes the neural model's value measurable, and hierarchy-consistent decoding guarantees structurally valid outputs.

The evaluation table above is the authoritative summary for the current run. Exact scores can vary slightly across TensorFlow builds and hardware despite deterministic seeds.